In [1]:
import subprocess
def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-2000:] if r.stdout else "")
    if r.stderr: print("ERR:", r.stderr[-300:])

run("pip install open-clip-torch==2.20.0 peft gdown SimpleITK pydicom tensorboardX -q")
run("git clone https://github.com/ChenJiayi68/DMTNet.git /kaggle/working/DMTNet")
run("git clone https://github.com/Lilacis/CD_TG.git /kaggle/working/CD_TG")
print("Done!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00

ERR: rotobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
ydf 0.14.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.


ERR: fatal: destination path '/kaggle/working/DMTNet' already exists and is not an empty directory.


ERR: fatal: destination path '/kaggle/working/CD_TG' already exists and is not an empty directory.

Done!


In [2]:
import gdown, os

# DMTNet weights
gdown.download(
    'https://drive.google.com/file/d/1FQFoHR_Y2qj5xN-YcCnxN8MePpITm38V/view?usp=sharing',
    '/kaggle/working/DMTNet/dmtnet_resnet50.pt', fuzzy=True)
print(f"DMTNet: {os.path.getsize('/kaggle/working/DMTNet/dmtnet_resnet50.pt')/1e6:.1f} MB")

# CLIP weights
gdown.download_folder(
    'https://drive.google.com/drive/folders/1BICJ2kXZe0TtgB9PrUNJRfE3IaPF_fJR',
    output='/kaggle/working/clip_weights', quiet=False)
print("CLIP:", os.listdir('/kaggle/working/clip_weights'))

Downloading...
From (original): https://drive.google.com/uc?id=1FQFoHR_Y2qj5xN-YcCnxN8MePpITm38V
From (redirected): https://drive.google.com/uc?id=1FQFoHR_Y2qj5xN-YcCnxN8MePpITm38V&confirm=t&uuid=bab230cd-5622-4546-b535-1be32c6a2f25
To: /kaggle/working/DMTNet/dmtnet_resnet50.pt
100%|██████████| 113M/113M [00:00<00:00, 138MB/s]  
Retrieving folder contents


DMTNet: 113.0 MB
Processing file 1VMDryounhIc-boJe4wRnFk24jNdg3gK6 RN50.pt
Processing file 1ep495t3VmPFlbrOTxZ8O7RjWzrUyNMom ViT-B-16.pt
Processing file 1tY28hHhNJ9UyqyVWATkedBkMdzzFPzsc ViT-B-32.pt


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1VMDryounhIc-boJe4wRnFk24jNdg3gK6
From (redirected): https://drive.google.com/uc?id=1VMDryounhIc-boJe4wRnFk24jNdg3gK6&confirm=t&uuid=b714f73f-d167-488b-b456-e2c95fd36780
To: /kaggle/working/clip_weights/RN50.pt
100%|██████████| 256M/256M [00:02<00:00, 121MB/s]  
Downloading...
From (original): https://drive.google.com/uc?id=1ep495t3VmPFlbrOTxZ8O7RjWzrUyNMom
From (redirected): https://drive.google.com/uc?id=1ep495t3VmPFlbrOTxZ8O7RjWzrUyNMom&confirm=t&uuid=893e7cbf-edb8-48ed-b9c0-9bc70fbb5a71
To: /kaggle/working/clip_weights/ViT-B-16.pt
100%|██████████| 351M/351M [00:04<00:00, 86.8MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1tY28hHhNJ9UyqyVWATkedBkMdzzFPzsc
From (redirected): https://drive.google.com/uc?id=1tY28hHhNJ9UyqyVWATkedBkMdzzFPzsc&confirm=t&uuid=06105a82-8b79-4995-8b72-fe659c76cc41
To: /k

CLIP: ['ViT-B-32.pt', 'ViT-B-16-openai.pt', 'RN50.pt', 'ViT-B-16.pt', 'ViT-B-16-converted.pt']


Download completed


In [4]:
# Fix 1: open_clip weights_only
factory_path = '/usr/local/lib/python3.12/dist-packages/open_clip/factory.py'
with open(factory_path) as f: c = f.read()
c = c.replace('weights_only: bool = True,', 'weights_only: bool = False,')
c = c.replace('weights_only=True,', 'weights_only=False,')
with open(factory_path, 'w') as f: f.write(c)
print("open_clip fixed ✓")

# Fix 2: encoder.py absolute path
encoder_path = '/kaggle/working/CD_TG/encoder.py'
with open(encoder_path) as f: c = f.read()
old = "category_file = category_file_map.get(model_name.lower(), 'clip_prompts/default.json')"
new = """_clip_dir = os.path.dirname(os.path.abspath(__file__))
        category_file = os.path.join(_clip_dir, category_file_map.get(model_name.lower(), 'clip_prompts/default.json'))"""
c = c.replace(old, new)
with open(encoder_path, 'w') as f: f.write(c)
print("encoder.py fixed ✓")

# Fix 3: logger.py add wbc and chaost2
logger_path = '/kaggle/working/DMTNet/common/logger.py'
with open(logger_path) as f: c = f.read()
old = """        elif self.benchmark == 'lung':
            self.class_ids_interest = dataset.class_ids
            self.class_ids_interest = torch.tensor(self.class_ids_interest).cuda()
            self.nclass = 1"""
new = """        elif self.benchmark == 'lung':
            self.class_ids_interest = dataset.class_ids
            self.class_ids_interest = torch.tensor(self.class_ids_interest).cuda()
            self.nclass = 1
        elif self.benchmark == 'wbc':
            self.class_ids_interest = dataset.class_ids
            self.class_ids_interest = torch.tensor(self.class_ids_interest).cuda()
            self.nclass = 1
        elif self.benchmark == 'chaost2':
            self.class_ids_interest = dataset.class_ids
            self.class_ids_interest = torch.tensor(self.class_ids_interest).cuda()
            self.nclass = 4"""
c = c.replace(old, new)
with open(logger_path, 'w') as f: f.write(c)
print("logger.py fixed ✓")


open_clip fixed ✓
encoder.py fixed ✓
logger.py fixed ✓


In [5]:
lung_code = '''r""" Chest X-ray few-shot semantic segmentation dataset """
import os
import glob
from torch.utils.data import Dataset
import torch.nn.functional as F
import torch
import PIL.Image as Image
import numpy as np


class DatasetLung(Dataset):
    def __init__(self, datapath, fold, transform, split, shot, num=600):
        self.split      = split
        self.nclass     = 1
        self.benchmark  = "lung"
        self.shot       = shot
        self.num        = num
        self.base_path  = os.path.join(datapath, "Lung Segmentation")
        self.img_path   = os.path.join(self.base_path, "CXR_png")
        self.ann_path   = os.path.join(self.base_path, "masks")
        self.transform  = transform
        self.categories = ["1"]
        self.class_ids  = range(0, 1)
        self.img_metadata_classwise = self.build_img_metadata_classwise()

    def __len__(self):
        return self.num

    def mask_to_img(self, mask_path):
        return os.path.basename(mask_path).replace("_mask.png", ".png")

    def __getitem__(self, idx):
        query_name, support_names, class_sample = self.sample_episode(idx)
        query_img, query_mask, support_imgs, support_masks = self.load_frame(
            query_name, support_names)
        query_img  = self.transform(query_img)
        query_mask = F.interpolate(
            query_mask.unsqueeze(0).unsqueeze(0).float(),
            query_img.size()[-2:], mode="nearest").squeeze()
        support_imgs = torch.stack([self.transform(s) for s in support_imgs])
        support_masks = torch.stack([
            F.interpolate(sm.unsqueeze(0).unsqueeze(0).float(),
                          support_imgs.size()[-2:], mode="nearest").squeeze()
            for sm in support_masks])
        return {
            "query_img":     query_img,
            "query_mask":    query_mask,
            "query_name":    query_name,
            "support_imgs":  support_imgs,
            "support_masks": support_masks,
            "support_names": support_names,
            "class_id":      torch.tensor(class_sample),
        }

    def load_frame(self, query_name, support_names):
        query_mask    = self.read_mask(query_name)
        support_masks = [self.read_mask(n) for n in support_names]
        query_img     = Image.open(
            os.path.join(self.img_path, self.mask_to_img(query_name))).convert("RGB")
        support_imgs  = [Image.open(
            os.path.join(self.img_path, self.mask_to_img(n))).convert("RGB")
            for n in support_names]
        return query_img, query_mask, support_imgs, support_masks

    def read_mask(self, img_name):
        mask = torch.tensor(np.array(Image.open(img_name).convert("L")))
        mask[mask < 128] = 0
        mask[mask >= 128] = 1
        return mask

    def sample_episode(self, idx):
        class_id     = idx % len(self.class_ids)
        class_sample = self.categories[class_id]
        query_name   = np.random.choice(
            self.img_metadata_classwise[class_sample], 1, replace=False)[0]
        support_names = []
        while True:
            s = np.random.choice(
                self.img_metadata_classwise[class_sample], 1, replace=False)[0]
            if s != query_name:
                support_names.append(s)
            if len(support_names) == self.shot:
                break
        return query_name, support_names, class_id

    def build_img_metadata_classwise(self):
        d = {"1": []}
        for p in sorted(glob.glob(os.path.join(self.ann_path, "*_mask.png"))):
            if os.path.exists(os.path.join(self.img_path, self.mask_to_img(p))):
                d["1"].append(p)
        print(f"Lung: {len(d[chr(49)])} matched pairs")
        return d
'''
with open('/kaggle/working/DMTNet/data/lung.py', 'w') as f:
    f.write(lung_code)
print("lung.py written ✓")

lung.py written ✓


In [6]:
wbc_code = '''r""" WBC few-shot segmentation dataset """
import os, glob, random
import numpy as np
import torch
import torch.nn.functional as F
import PIL.Image as Image
from torch.utils.data import Dataset


class DatasetWBC(Dataset):
    def __init__(self, datapath, fold, transform, split, shot=1, use_original_imgsize=False):
        self.shot      = shot
        self.transform = transform
        self.benchmark = "wbc"
        self.split     = split
        self.base_path = os.path.join(datapath, "wbc", "segmentation_WBC-master")
        self.img_path  = os.path.join(self.base_path, "images")
        self.ann_path  = os.path.join(self.base_path, "masks")
        self.categories = ["1"]
        self.class_ids  = [0]
        self.img_metadata_classwise = self._build_metadata()
        print(f"WBC: {len(self.img_metadata_classwise[chr(49)])} pairs found")

    def _mask_to_img(self, p):
        return p.replace(".png", ".bmp").replace("masks", "images")

    def _build_metadata(self):
        d = {"1": []}
        for p in sorted(glob.glob(os.path.join(self.ann_path, "*.png"))):
            if os.path.exists(self._mask_to_img(p)):
                d["1"].append(p)
        return d

    def __len__(self):
        return 1000

    def read_mask(self, path):
        mask = torch.tensor(np.array(Image.open(path).convert("L")))
        return (mask >= 50).long()

    def __getitem__(self, idx):
        all_masks = self.img_metadata_classwise["1"]
        chosen    = random.sample(all_masks, 1 + self.shot)
        q_msk, s_msks = chosen[0], chosen[1:]
        q_img  = self.transform(Image.open(self._mask_to_img(q_msk)).convert("RGB"))
        q_mask = self.read_mask(q_msk)
        q_mask = F.interpolate(q_mask.unsqueeze(0).unsqueeze(0).float(),
                               q_img.size()[-2:], mode="nearest").squeeze().long()
        s_imgs, s_masks = [], []
        for sm in s_msks:
            si   = self.transform(Image.open(self._mask_to_img(sm)).convert("RGB"))
            sm_t = F.interpolate(self.read_mask(sm).unsqueeze(0).unsqueeze(0).float(),
                                 si.size()[-2:], mode="nearest").squeeze().long()
            s_imgs.append(si)
            s_masks.append(sm_t)
        return {
            "query_img":     q_img,
            "query_mask":    q_mask,
            "query_name":    q_msk,
            "support_imgs":  torch.stack(s_imgs),
            "support_masks": torch.stack(s_masks),
            "support_names": s_msks,
            "class_id":      torch.tensor(0),
        }
'''
with open('/kaggle/working/DMTNet/data/wbc.py', 'w') as f:
    f.write(wbc_code)
print("wbc.py written ✓")

wbc.py written ✓


In [6]:
chaos_code = '''r""" CHAOS-MRI T2SPIR few-shot segmentation dataset """
import os, glob, random
import numpy as np
import torch
import torch.nn.functional as F
import PIL.Image as Image
import SimpleITK as sitk
from torch.utils.data import Dataset

ORGAN_MAP = {"liver": 63, "rk": 126, "lk": 189, "spleen": 252}

class DatasetCHAOST2(Dataset):
    def __init__(self, datapath, fold, transform, split, shot=1, use_original_imgsize=False):
        self.shot       = shot
        self.transform  = transform
        self.benchmark  = "chaost2"
        self.split      = split
        self.base_path  = os.path.join(datapath, "chaos", "chaos_train", "Train_Sets", "MR")
        self.categories = list(ORGAN_MAP.keys())
        self.class_ids  = list(range(len(self.categories)))
        self.metadata   = self._build_metadata()
        total = sum(len(v) for v in self.metadata.values())
        print(f"CHAOS: {total} slices found")

    def _build_metadata(self):
        d = {o: [] for o in ORGAN_MAP}
        patients = sorted(os.listdir(self.base_path))
        n = len(patients)
        chosen = patients[int(0.8*n):] if self.split == "test" else patients[:int(0.8*n)]
        for pid in chosen:
            dcm_dir  = os.path.join(self.base_path, pid, "T2SPIR", "DICOM_anon")
            mask_dir = os.path.join(self.base_path, pid, "T2SPIR", "Ground")
            if not os.path.isdir(dcm_dir): continue
            dcms  = sorted(glob.glob(os.path.join(dcm_dir, "*.dcm")))
            masks = sorted(glob.glob(os.path.join(mask_dir, "*.png")))
            for dcm, msk in zip(dcms, masks):
                arr = np.array(Image.open(msk).convert("L"))
                for organ, val in ORGAN_MAP.items():
                    if (arr == val).sum() >= 50:
                        d[organ].append((dcm, msk, val))
        return d

    def _load_dicom(self, path):
        img = sitk.GetArrayFromImage(sitk.ReadImage(path))[0].astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-6) * 255
        return Image.fromarray(img.astype(np.uint8)).convert("RGB")

    def __len__(self):
        return 1000

    def __getitem__(self, idx):
        organ   = random.choice(self.categories)
        samples = self.metadata[organ]
        chosen  = random.sample(samples, 1 + self.shot)
        q_dcm, q_msk, val = chosen[0]
        s_items = chosen[1:]
        q_img  = self.transform(self._load_dicom(q_dcm))
        q_mask = torch.tensor(
            (np.array(Image.open(q_msk).convert("L")) == val).astype(np.uint8)).long()
        q_mask = F.interpolate(q_mask.unsqueeze(0).unsqueeze(0).float(),
                               q_img.size()[-2:], mode="nearest").squeeze().long()
        s_imgs, s_masks = [], []
        for (s_dcm, s_msk, s_val) in s_items:
            si = self.transform(self._load_dicom(s_dcm))
            sm = torch.tensor(
                (np.array(Image.open(s_msk).convert("L")) == s_val).astype(np.uint8)).long()
            sm = F.interpolate(sm.unsqueeze(0).unsqueeze(0).float(),
                               si.size()[-2:], mode="nearest").squeeze().long()
            s_imgs.append(si)
            s_masks.append(sm)
        return {
            "query_img":     q_img,
            "query_mask":    q_mask,
            "query_name":    q_dcm,
            "support_imgs":  torch.stack(s_imgs),
            "support_masks": torch.stack(s_masks),
            "support_names": [s[0] for s in s_items],
            "class_id":      torch.tensor(self.categories.index(organ)),
        }
'''
with open('/kaggle/working/DMTNet/data/chaost2.py', 'w') as f:
    f.write(chaos_code)
print("chaost2.py written ✓")

chaost2.py written ✓


In [8]:
dataset_code = '''r""" Dataloader builder """
import torch
from torchvision import transforms
from torch.utils.data import DataLoader
from data.pascal import DatasetPASCAL
from data.fss import DatasetFSS
from data.deepglobe import DatasetDeepglobe
from data.isic import DatasetISIC
from data.lung import DatasetLung
from data.wbc import DatasetWBC
from data.chaost2 import DatasetCHAOST2


class FSSDataset:
    @classmethod
    def initialize(cls, img_size, datapath):
        cls.datasets = {
            "pascal":    DatasetPASCAL,
            "fss":       DatasetFSS,
            "deepglobe": DatasetDeepglobe,
            "isic":      DatasetISIC,
            "lung":      DatasetLung,
            "wbc":       DatasetWBC,
            "chaost2":   DatasetCHAOST2,
        }
        cls.img_mean = [0.485, 0.456, 0.406]
        cls.img_std  = [0.229, 0.224, 0.225]
        cls.datapath = datapath
        cls.transform = transforms.Compose([
            transforms.Resize(size=(img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(cls.img_mean, cls.img_std)])

    @classmethod
    def build_dataloader(cls, benchmark, bsz, nworker, fold, split, shot=1):
        nworker = nworker if split == "trn" else 0
        dataset = cls.datasets[benchmark](
            cls.datapath, fold=fold, transform=cls.transform,
            split=split, shot=shot)
        dataloader = DataLoader(dataset, batch_size=bsz,
                                shuffle=(split == "trn"),
                                num_workers=nworker,
                                drop_last=False,
                                pin_memory=True)
        return dataloader, None
'''
with open('/kaggle/working/DMTNet/data/datasetDDP.py', 'w') as f:
    f.write(dataset_code)
print("datasetDDP.py written ✓")

datasetDDP.py written ✓


In [9]:
import sys, torch, torch.nn as nn
sys.path.insert(0, '/kaggle/working/DMTNet')
sys.path.insert(0, '/kaggle/working/CD_TG')

from model.dmtnet import DMTNetwork
from common.evaluation import Evaluator
from common import utils
from common.logger import AverageMeter
from data.datasetDDP import FSSDataset

device = torch.device('cuda')

model = DMTNetwork('resnet50')
src = torch.load('/kaggle/working/DMTNet/dmtnet_resnet50.pt',
                 map_location=device, weights_only=False)
model = nn.DataParallel(model)
model.to(device)
model.load_state_dict(src, strict=False)
model.eval()
print("Model loaded ✓")
Evaluator.initialize()

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 191MB/s] 


Model loaded ✓


In [10]:
utils.fix_randseed(0)
FSSDataset.initialize(img_size=400,
    datapath='/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels')
dl, _ = FSSDataset.build_dataloader('lung', 20, 0, 0, 'test', 1)
avg = AverageMeter(dl.dataset)

with torch.no_grad():
    for idx, batch in enumerate(dl):
        batch = utils.to_cuda(batch)
        pred_mask, _, _ = model.module.predict_mask_nshot(batch, nshot=1)
        assert pred_mask.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred_mask.clone(), batch)
        avg.update(inter, union, batch['class_id'], None)
        if idx % 5 == 0:
            m, _ = avg.compute_iou()
            print(f"Batch {idx+1}/{len(dl)} mIoU: {m:.2f}", flush=True)

miou_lung_1s, _ = avg.compute_iou()
print(f"\nLung 1-shot: {miou_lung_1s:.2f}  (paper: 73.7)")

Lung: 566 matched pairs
Batch 1/30 mIoU: 61.94
Batch 6/30 mIoU: 64.23
Batch 11/30 mIoU: 64.30
Batch 16/30 mIoU: 63.71
Batch 21/30 mIoU: 63.49
Batch 26/30 mIoU: 63.20

Lung 1-shot: 63.29  (paper: 73.7)


In [12]:
utils.fix_randseed(0)
dl, _ = FSSDataset.build_dataloader('lung', 8, 0, 0, 'test', 5)
avg = AverageMeter(dl.dataset)

with torch.no_grad():
    for idx, batch in enumerate(dl):
        batch = utils.to_cuda(batch)
        pred_mask, _, _ = model.module.predict_mask_nshot(batch, nshot=5)
        assert pred_mask.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred_mask.clone(), batch)
        avg.update(inter, union, batch['class_id'], None)
        if idx % 5 == 0:
            m, _ = avg.compute_iou()
            print(f"Batch {idx+1}/{len(dl)} mIoU: {m:.2f}", flush=True)

miou_lung_5s, _ = avg.compute_iou()
print(f"\nLung 5-shot: {miou_lung_5s:.2f}  (paper: 77.3)")

Lung: 566 matched pairs
Batch 1/75 mIoU: 70.25
Batch 6/75 mIoU: 62.43
Batch 11/75 mIoU: 64.80
Batch 16/75 mIoU: 64.76
Batch 21/75 mIoU: 64.94
Batch 26/75 mIoU: 64.92
Batch 31/75 mIoU: 64.47
Batch 36/75 mIoU: 64.49
Batch 41/75 mIoU: 64.96
Batch 46/75 mIoU: 65.28
Batch 51/75 mIoU: 65.29
Batch 56/75 mIoU: 65.08
Batch 61/75 mIoU: 64.97
Batch 66/75 mIoU: 64.75
Batch 71/75 mIoU: 64.71

Lung 5-shot: 64.71  (paper: 77.3)


In [13]:
with open('/kaggle/working/DMTNet/data/isic.py') as f:
    print(f.read())

r""" ISIC few-shot semantic segmentation dataset """
import os
import glob

from torch.utils.data import Dataset
import torch.nn.functional as F
import torch
import PIL.Image as Image
import numpy as np

# 1:1867 2:519 3:208
class DatasetISIC(Dataset):
    def __init__(self, datapath, fold, transform, split, shot, num=600):
        self.split = split
        self.benchmark = 'isic'
        self.shot = shot
        if self.split == 'test':
            self.num = num
            self.categories = ['1','2','3']

        self.base_path = os.path.join(datapath, 'ISIC')
        self.img_path = os.path.join(self.base_path, 'ISIC2018_Task1-2_Training_Input')
        self.ann_path = os.path.join(self.base_path, 'ISIC2018_Task1_Training_GroundTruth')
        self.transform = transform

        self.class_ids = range(0, 3)
        self.img_metadata_classwise = self.build_img_metadata_classwise()       

    def __len__(self):
        return self.num

    def __getitem__(self, idx):
        query_

In [14]:
import os, glob

isic_base = '/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation'
for root, dirs, files in os.walk(isic_base):
    depth = root.replace(isic_base, '').count(os.sep)
    if depth <= 2:
        print(f"{root} → {len(files)} files")
        if files:
            print(f"  sample: {files[:3]}")

/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation → 0 files
/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation/ISIC2018_Task1-2_Training_Input → 2596 files
  sample: ['ISIC_0012706.jpg', 'ISIC_0010192.jpg', 'ISIC_0010017.jpg']
/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation/ISIC2018_Task1_Training_GroundTruth → 2596 files
  sample: ['ISIC_0000505_segmentation.png', 'ISIC_0015129_segmentation.png', 'ISIC_0014743_segmentation.png']
/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation/ISIC2018_Task1-2_Test_Input → 1002 files
  sample: ['ISIC_0036343.jpg', 'ISIC_0023300.jpg', 'ISIC_0019816.jpg']
/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation/ISIC2018_Task1-2_Validation_Input → 102 files
  sample: ['ISIC_0017341.jpg', 'ISIC_0015480.jpg', 'ISIC_0015518.jpg']
/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation/ISIC2018_Task1-2_Validation_Input/.i

In [15]:
import os
print(os.listdir('/kaggle/working/CD_TG/dataloader'))
# Check for ISIC split file
isic_split = '/kaggle/working/CD_TG/dataloader/ISIC_Split.py'
if os.path.exists(isic_split):
    with open(isic_split) as f:
        print(f.read())

['coco.py', 'dataset_specifics.py', 'isic.py', 'dataset.py', 'wbc.py', 'isic', 'splits', 'ISIC_Split.py', 'pascal.py', 'lung.py', 'chaost2.py']
import pandas as pd
import os
from PIL import Image

df = pd.read_csv('.../isic/class_id.csv')

dir = '.../dataset/ISIC/ISIC2018_Task1-2_Training_Input/'

dict = {'seborrheic_keratosis' : '3','nevus' : '2','melanoma' : '1'}

for directory in ['1', '2', '3']:
    subdir = os.path.join(dir,directory)
    if not os.path.exists(subdir):
        os.makedirs(subdir)

for idx, row in df.iterrows():

    src_path = dir + row['ID'] + '.jpg'
    dest_path = dir + dict[row['Class']] + '/' + row['ID'] + '.jpg'

    try:
        img = Image.open(src_path)
        img_resized = img.resize((512, 512))
        img_resized.save(dest_path)
    except:
        continue



In [16]:
import os
# Check CD_TG isic folder
print(os.listdir('/kaggle/working/CD_TG/dataloader/isic'))

['class_id.csv']


In [17]:
import pandas as pd
import os, shutil, glob

ISIC_INPUT  = '/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation'
ISIC_WORK   = '/kaggle/working/ISIC'
CLASS_CSV   = '/kaggle/working/CD_TG/dataloader/isic/class_id.csv'

img_src  = os.path.join(ISIC_INPUT, 'ISIC2018_Task1-2_Training_Input')
mask_src = os.path.join(ISIC_INPUT, 'ISIC2018_Task1_Training_GroundTruth')

# Create destination folders
for d in ['1','2','3']:
    os.makedirs(os.path.join(ISIC_WORK, 'ISIC2018_Task1-2_Training_Input', d), exist_ok=True)
os.makedirs(os.path.join(ISIC_WORK, 'ISIC2018_Task1_Training_GroundTruth'), exist_ok=True)

# Read class CSV
df = pd.read_csv(CLASS_CSV)
print(f"CSV loaded: {len(df)} rows")
print(df.head())
print(f"Columns: {df.columns.tolist()}")


CSV loaded: 2594 rows
             ID     Class
0  ISIC_0000000     nevus
1  ISIC_0000001     nevus
2  ISIC_0000003     nevus
3  ISIC_0000004  melanoma
4  ISIC_0000006     nevus
Columns: ['ID', 'Class']


In [18]:
import pandas as pd
import os, shutil

ISIC_INPUT = '/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation'
ISIC_WORK  = '/kaggle/working/ISIC'
CLASS_CSV  = '/kaggle/working/CD_TG/dataloader/isic/class_id.csv'

img_src  = os.path.join(ISIC_INPUT, 'ISIC2018_Task1-2_Training_Input')
mask_src = os.path.join(ISIC_INPUT, 'ISIC2018_Task1_Training_GroundTruth')

class_map = {'melanoma': '1', 'nevus': '2', 'seborrheic_keratosis': '3'}

# Create folders
for d in ['1','2','3']:
    os.makedirs(os.path.join(ISIC_WORK, 'ISIC2018_Task1-2_Training_Input', d), exist_ok=True)
os.makedirs(os.path.join(ISIC_WORK, 'ISIC2018_Task1_Training_GroundTruth'), exist_ok=True)

df = pd.read_csv(CLASS_CSV)
copied_imgs  = 0
copied_masks = 0
skipped      = 0

for _, row in df.iterrows():
    img_id    = row['ID']
    class_str = row['Class']
    folder    = class_map.get(class_str)
    if folder is None:
        skipped += 1
        continue

    # Copy image
    src_img  = os.path.join(img_src, img_id + '.jpg')
    dst_img  = os.path.join(ISIC_WORK, 'ISIC2018_Task1-2_Training_Input', folder, img_id + '.jpg')
    if os.path.exists(src_img):
        shutil.copy(src_img, dst_img)
        copied_imgs += 1

    # Copy mask
    src_mask = os.path.join(mask_src, img_id + '_segmentation.png')
    dst_mask = os.path.join(ISIC_WORK, 'ISIC2018_Task1_Training_GroundTruth', img_id + '_segmentation.png')
    if os.path.exists(src_mask):
        shutil.copy(src_mask, dst_mask)
        copied_masks += 1

print(f"Images copied:  {copied_imgs}")
print(f"Masks copied:   {copied_masks}")
print(f"Skipped:        {skipped}")
print(f"\nFolder counts:")
for d in ['1','2','3']:
    n = len(os.listdir(os.path.join(ISIC_WORK, 'ISIC2018_Task1-2_Training_Input', d)))
    print(f"  Class {d}: {n} images")

Images copied:  2594
Masks copied:   2594
Skipped:        0

Folder counts:
  Class 1: 519 images
  Class 2: 1867 images
  Class 3: 208 images


In [19]:
import importlib, sys
for mod in list(sys.modules.keys()):
    if 'isic' in mod or 'dataset' in mod or 'data.' in mod:
        del sys.modules[mod]

FSSDataset.initialize(img_size=400, datapath='/kaggle/working')
dl, _ = FSSDataset.build_dataloader('isic', 20, 0, 0, 'test', 1)
batch = next(iter(dl))
print(f"ISIC dataloader ready: {len(dl)} batches")
print(f"query_img shape: {batch['query_img'].shape}")
print(f"query_mask unique: {batch['query_mask'].unique()}")

Total (test) isic images are : 600
ISIC dataloader ready: 30 batches
query_img shape: torch.Size([20, 3, 400, 400])
query_mask unique: tensor([0., 1.])


In [21]:
import importlib, sys
for mod in list(sys.modules.keys()):
    if 'isic' in mod or 'dataset' in mod or 'data.' in mod:
        del sys.modules[mod]

FSSDataset.initialize(img_size=400, datapath='/kaggle/working')
dl, _ = FSSDataset.build_dataloader('isic', 20, 0, 0, 'test', 1)
batch = next(iter(dl))
print(f"ISIC dataloader ready: {len(dl)} batches")
print(f"query_img shape: {batch['query_img'].shape}")
print(f"query_mask unique: {batch['query_mask'].unique()}")


Total (test) isic images are : 600
ISIC dataloader ready: 30 batches
query_img shape: torch.Size([20, 3, 400, 400])
query_mask unique: tensor([0., 1.])


In [25]:
import torch, gc

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()
print(f"GPU memory free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

GPU memory free: 0.13 GB


In [26]:
utils.fix_randseed(0)
FSSDataset.initialize(img_size=400, datapath='/kaggle/working')
dl, _ = FSSDataset.build_dataloader('isic', 10, 0, 0, 'test', 1)
avg = AverageMeter(dl.dataset)

with torch.no_grad():
    for idx, batch in enumerate(dl):
        batch = utils.to_cuda(batch)
        pred_mask, _, _ = model.module.predict_mask_nshot(batch, nshot=1)
        assert pred_mask.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred_mask.clone(), batch)
        avg.update(inter, union, batch['class_id'], None)
        if idx % 5 == 0:
            m, _ = avg.compute_iou()
            print(f"Batch {idx+1}/{len(dl)} mIoU: {m:.2f}", flush=True)

miou_isic_1s, _ = avg.compute_iou()
print(f"\nISIC 1-shot: {miou_isic_1s:.2f}  (paper: 40.1)")

Total (test) isic images are : 600
Batch 1/60 mIoU: 55.31
Batch 6/60 mIoU: 40.64
Batch 11/60 mIoU: 42.90
Batch 16/60 mIoU: 41.29
Batch 21/60 mIoU: 43.15
Batch 26/60 mIoU: 42.54
Batch 31/60 mIoU: 42.04
Batch 36/60 mIoU: 42.84
Batch 41/60 mIoU: 42.72
Batch 46/60 mIoU: 42.66
Batch 51/60 mIoU: 43.12
Batch 56/60 mIoU: 43.72

ISIC 1-shot: 43.97  (paper: 40.1)


In [27]:
utils.fix_randseed(0)
dl, _ = FSSDataset.build_dataloader('isic', 4, 0, 0, 'test', 5)
avg = AverageMeter(dl.dataset)

with torch.no_grad():
    for idx, batch in enumerate(dl):
        batch = utils.to_cuda(batch)
        pred_mask, _, _ = model.module.predict_mask_nshot(batch, nshot=5)
        assert pred_mask.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred_mask.clone(), batch)
        avg.update(inter, union, batch['class_id'], None)
        if idx % 10 == 0:
            m, _ = avg.compute_iou()
            print(f"Batch {idx+1}/{len(dl)} mIoU: {m:.2f}", flush=True)

miou_isic_5s, _ = avg.compute_iou()
print(f"\nISIC 5-shot: {miou_isic_5s:.2f}  (paper: 51.1)")

Total (test) isic images are : 600
Batch 1/150 mIoU: 71.32
Batch 11/150 mIoU: 50.29
Batch 21/150 mIoU: 52.30
Batch 31/150 mIoU: 51.54
Batch 41/150 mIoU: 51.53
Batch 51/150 mIoU: 51.03
Batch 61/150 mIoU: 51.00
Batch 71/150 mIoU: 52.22
Batch 81/150 mIoU: 52.09
Batch 91/150 mIoU: 51.45
Batch 101/150 mIoU: 51.64
Batch 111/150 mIoU: 52.09
Batch 121/150 mIoU: 52.58
Batch 131/150 mIoU: 51.62
Batch 141/150 mIoU: 51.46

ISIC 5-shot: 51.96  (paper: 51.1)


In [28]:
import os, shutil, glob

# Set this to your actual Kaggle WBC input path
WBC_INPUT = '/kaggle/input/datasets/satvikagada/wbcdataset/segmentation_WBC-master'
WBC_WORK  = '/kaggle/working/wbc_organized/segmentation_WBC-master'

# Create output folders
os.makedirs(os.path.join(WBC_WORK, 'images'), exist_ok=True)
os.makedirs(os.path.join(WBC_WORK, 'masks'),  exist_ok=True)

# Copy Dataset 1 with d1_ prefix
for f in sorted(glob.glob(os.path.join(WBC_INPUT, 'Dataset 1', '*.bmp'))):
    shutil.copy(f, os.path.join(WBC_WORK, 'images', 'd1_' + os.path.basename(f)))
for f in sorted(glob.glob(os.path.join(WBC_INPUT, 'Dataset 1', '*.png'))):
    shutil.copy(f, os.path.join(WBC_WORK, 'masks',  'd1_' + os.path.basename(f)))

# Copy Dataset 2 with d2_ prefix
for f in sorted(glob.glob(os.path.join(WBC_INPUT, 'Dataset 2', '*.bmp'))):
    shutil.copy(f, os.path.join(WBC_WORK, 'images', 'd2_' + os.path.basename(f)))
for f in sorted(glob.glob(os.path.join(WBC_INPUT, 'Dataset 2', '*.png'))):
    shutil.copy(f, os.path.join(WBC_WORK, 'masks',  'd2_' + os.path.basename(f)))

imgs  = glob.glob(os.path.join(WBC_WORK, 'images', '*.bmp'))
masks = glob.glob(os.path.join(WBC_WORK, 'masks',  '*.png'))
print(f"Images: {len(imgs)}, Masks: {len(masks)}")

Images: 400, Masks: 400


In [29]:
import torch, gc

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()
print(f"GPU memory free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

GPU memory free: 3.49 GB


In [38]:
import importlib, sys, glob, os

# Clear cached data modules
for mod in list(sys.modules.keys()):
    if 'wbc' in mod or 'dataset' in mod or 'data.' in mod:
        del sys.modules[mod]

# Verify organized folder has files
wbc_work = '/kaggle/working/wbc_organized/segmentation_WBC-master'
imgs  = glob.glob(os.path.join(wbc_work, 'images', '*.bmp'))
masks = glob.glob(os.path.join(wbc_work, 'masks', '*.png'))
print(f"Images: {len(imgs)}, Masks: {len(masks)}")

# Reimport and reinitialize
from data.datasetDDP import FSSDataset
from common.evaluation import Evaluator
Evaluator.initialize()
FSSDataset.initialize(img_size=400, datapath='/kaggle/working/wbc_organized')
dl, _ = FSSDataset.build_dataloader('wbc', 10, 0, 0, 'test', 1)
print(f"WBC dataloader: {len(dl)} batches")

# Quick check dataset has pairs
print(f"WBC pairs: {len(dl.dataset.img_metadata_classwise['1'])}")


Images: 400, Masks: 400
WBC: 0 pairs found
WBC dataloader: 100 batches
WBC pairs: 0


In [39]:
import glob, os

wbc_work = '/kaggle/working/wbc_organized/segmentation_WBC-master'
masks = sorted(glob.glob(os.path.join(wbc_work, 'masks', '*.png')))[:3]
print("Sample mask paths:")
for m in masks:
    print(f"  mask: {m}")
    img = m.replace('.png', '.bmp').replace('masks', 'images')
    print(f"  img:  {img}")
    print(f"  exists: {os.path.exists(img)}")

Sample mask paths:
  mask: /kaggle/working/wbc_organized/segmentation_WBC-master/masks/d1_001.png
  img:  /kaggle/working/wbc_organized/segmentation_WBC-master/images/d1_001.bmp
  exists: True
  mask: /kaggle/working/wbc_organized/segmentation_WBC-master/masks/d1_002.png
  img:  /kaggle/working/wbc_organized/segmentation_WBC-master/images/d1_002.bmp
  exists: True
  mask: /kaggle/working/wbc_organized/segmentation_WBC-master/masks/d1_003.png
  img:  /kaggle/working/wbc_organized/segmentation_WBC-master/images/d1_003.bmp
  exists: True


In [40]:
wbc_code = '''r""" WBC few-shot segmentation dataset """
import os, glob, random
import numpy as np
import torch
import torch.nn.functional as F
import PIL.Image as Image
from torch.utils.data import Dataset


class DatasetWBC(Dataset):
    def __init__(self, datapath, fold, transform, split, shot=1, use_original_imgsize=False):
        self.shot      = shot
        self.transform = transform
        self.benchmark = "wbc"
        self.split     = split

        # Try both path structures
        candidate1 = os.path.join(datapath, "wbc", "segmentation_WBC-master")
        candidate2 = os.path.join(datapath, "segmentation_WBC-master")
        candidate3 = datapath

        if os.path.exists(os.path.join(candidate1, "images")):
            base = candidate1
        elif os.path.exists(os.path.join(candidate2, "images")):
            base = candidate2
        elif os.path.exists(os.path.join(candidate3, "images")):
            base = candidate3
        else:
            base = candidate1  # fallback

        self.img_path  = os.path.join(base, "images")
        self.ann_path  = os.path.join(base, "masks")
        self.categories = ["1"]
        self.class_ids  = [0]
        self.img_metadata_classwise = self._build_metadata()
        print(f"WBC: {len(self.img_metadata_classwise[chr(49)])} pairs found")
        print(f"WBC img_path: {self.img_path}")

    def _mask_to_img(self, p):
        return p.replace(".png", ".bmp").replace("masks", "images")

    def _build_metadata(self):
        d = {"1": []}
        for p in sorted(glob.glob(os.path.join(self.ann_path, "*.png"))):
            if os.path.exists(self._mask_to_img(p)):
                d["1"].append(p)
        return d

    def __len__(self):
        return 1000

    def read_mask(self, path):
        mask = torch.tensor(np.array(Image.open(path).convert("L")))
        return (mask >= 50).long()

    def __getitem__(self, idx):
        all_masks = self.img_metadata_classwise["1"]
        chosen    = random.sample(all_masks, 1 + self.shot)
        q_msk, s_msks = chosen[0], chosen[1:]
        q_img  = self.transform(Image.open(self._mask_to_img(q_msk)).convert("RGB"))
        q_mask = self.read_mask(q_msk)
        q_mask = F.interpolate(q_mask.unsqueeze(0).unsqueeze(0).float(),
                               q_img.size()[-2:], mode="nearest").squeeze().long()
        s_imgs, s_masks = [], []
        for sm in s_msks:
            si   = self.transform(Image.open(self._mask_to_img(sm)).convert("RGB"))
            sm_t = F.interpolate(self.read_mask(sm).unsqueeze(0).unsqueeze(0).float(),
                                 si.size()[-2:], mode="nearest").squeeze().long()
            s_imgs.append(si)
            s_masks.append(sm_t)
        return {
            "query_img":     q_img,
            "query_mask":    q_mask,
            "query_name":    q_msk,
            "support_imgs":  torch.stack(s_imgs),
            "support_masks": torch.stack(s_masks),
            "support_names": s_msks,
            "class_id":      torch.tensor(0),
        }
'''
with open('/kaggle/working/DMTNet/data/wbc.py', 'w') as f:
    f.write(wbc_code)
print("wbc.py fixed ✓")

wbc.py fixed ✓


In [42]:
for mod in list(sys.modules.keys()):
    if 'wbc' in mod or 'dataset' in mod or 'data.' in mod:
        del sys.modules[mod]

from data.datasetDDP import FSSDataset
FSSDataset.initialize(img_size=400,
    datapath='/kaggle/working/wbc_organized/segmentation_WBC-master')
dl, _ = FSSDataset.build_dataloader('wbc', 10, 0, 0, 'test', 1)
print(f"Pairs: {len(dl.dataset.img_metadata_classwise['1'])}")
print(f"Batches: {len(dl)}")


WBC: 400 pairs found
WBC img_path: /kaggle/working/wbc_organized/segmentation_WBC-master/images
Pairs: 400
Batches: 100


In [43]:
utils.fix_randseed(0)
FSSDataset.initialize(img_size=400, datapath='/kaggle/working/wbc_organized')
dl, _ = FSSDataset.build_dataloader('wbc', 10, 0, 0, 'test', 1)
avg = AverageMeter(dl.dataset)

with torch.no_grad():
    for idx, batch in enumerate(dl):
        batch = utils.to_cuda(batch)
        pred_mask, _, _ = model.module.predict_mask_nshot(batch, nshot=1)
        assert pred_mask.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred_mask.clone(), batch)
        avg.update(inter, union, batch['class_id'], None)
        if idx % 10 == 0:
            m, _ = avg.compute_iou()
            print(f"Batch {idx+1}/{len(dl)} mIoU: {m:.2f}", flush=True)

miou_wbc_1s, _ = avg.compute_iou()
print(f"\nWBC 1-shot: {miou_wbc_1s:.2f}  (paper: 48.1)")

WBC: 400 pairs found
WBC img_path: /kaggle/working/wbc_organized/segmentation_WBC-master/images
Batch 1/100 mIoU: 83.05
Batch 11/100 mIoU: 80.15
Batch 21/100 mIoU: 78.38
Batch 31/100 mIoU: 79.69
Batch 41/100 mIoU: 78.86
Batch 51/100 mIoU: 78.13
Batch 61/100 mIoU: 78.00
Batch 71/100 mIoU: 78.32
Batch 81/100 mIoU: 78.44
Batch 91/100 mIoU: 78.65

WBC 1-shot: 79.19  (paper: 48.1)


In [44]:
torch.cuda.empty_cache()
gc.collect()

utils.fix_randseed(0)
dl, _ = FSSDataset.build_dataloader('wbc', 4, 0, 0, 'test', 5)
avg = AverageMeter(dl.dataset)

with torch.no_grad():
    for idx, batch in enumerate(dl):
        batch = utils.to_cuda(batch)
        pred_mask, _, _ = model.module.predict_mask_nshot(batch, nshot=5)
        assert pred_mask.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred_mask.clone(), batch)
        avg.update(inter, union, batch['class_id'], None)
        if idx % 10 == 0:
            m, _ = avg.compute_iou()
            print(f"Batch {idx+1}/{len(dl)} mIoU: {m:.2f}", flush=True)

miou_wbc_5s, _ = avg.compute_iou()
print(f"\nWBC 5-shot: {miou_wbc_5s:.2f}  (paper: 49.5)")

WBC: 400 pairs found
WBC img_path: /kaggle/working/wbc_organized/segmentation_WBC-master/images
Batch 1/250 mIoU: 75.32
Batch 11/250 mIoU: 84.56
Batch 21/250 mIoU: 82.80
Batch 31/250 mIoU: 83.52
Batch 41/250 mIoU: 83.93
Batch 51/250 mIoU: 84.29
Batch 61/250 mIoU: 84.15
Batch 71/250 mIoU: 83.89
Batch 81/250 mIoU: 84.13
Batch 91/250 mIoU: 83.90
Batch 101/250 mIoU: 84.32
Batch 111/250 mIoU: 84.49
Batch 121/250 mIoU: 84.98
Batch 131/250 mIoU: 84.80
Batch 141/250 mIoU: 84.60
Batch 151/250 mIoU: 84.83
Batch 161/250 mIoU: 84.82
Batch 171/250 mIoU: 84.91
Batch 181/250 mIoU: 85.07
Batch 191/250 mIoU: 84.91
Batch 201/250 mIoU: 84.95
Batch 211/250 mIoU: 84.79
Batch 221/250 mIoU: 84.87
Batch 231/250 mIoU: 84.79
Batch 241/250 mIoU: 84.97

WBC 5-shot: 84.96  (paper: 49.5)


In [10]:
with open('/kaggle/working/DMTNet/data/chaost2.py') as f:
    c = f.read()

# Remove old path and set correct one
c = c.replace(
    'os.path.join(datapath, "chaos", "chaos_train", "Train_Sets", "MR")',
    'os.path.join(datapath, "CHAOS_Train_Sets", "MR")'
)
# Also handle if it still has old path
c = c.replace(
    'os.path.join(datapath, "CHAOS_Train_Sets", "MR")',
    'os.path.join(datapath, "CHAOS_Train_Sets", "MR")'
)

with open('/kaggle/working/DMTNet/data/chaost2.py', 'w') as f:
    f.write(c)
print("Fixed ✓")

# Verify
with open('/kaggle/working/DMTNet/data/chaost2.py') as f:
    for line in f:
        if 'base_path' in line or 'MR' in line:
            print(line.rstrip())


Fixed ✓
r""" CHAOS-MRI T2SPIR few-shot segmentation dataset """
        self.base_path  = os.path.join(datapath, "CHAOS_Train_Sets", "MR")
        patients = sorted(os.listdir(self.base_path))
            dcm_dir  = os.path.join(self.base_path, pid, "T2SPIR", "DICOM_anon")
            mask_dir = os.path.join(self.base_path, pid, "T2SPIR", "Ground")


In [11]:
with open('/kaggle/working/DMTNet/data/chaost2.py') as f:
    c = f.read()

# Replace whatever path is there with correct one
import re
c = re.sub(
    r'self\.base_path\s*=.*',
    'self.base_path  = os.path.join(datapath, "Train_Sets", "MR")',
    c
)

with open('/kaggle/working/DMTNet/data/chaost2.py', 'w') as f:
    f.write(c)

# Verify
with open('/kaggle/working/DMTNet/data/chaost2.py') as f:
    for line in f:
        if 'base_path' in line:
            print(line.rstrip())


        self.base_path  = os.path.join(datapath, "Train_Sets", "MR")
        patients = sorted(os.listdir(self.base_path))
            dcm_dir  = os.path.join(self.base_path, pid, "T2SPIR", "DICOM_anon")
            mask_dir = os.path.join(self.base_path, pid, "T2SPIR", "Ground")


In [12]:
for mod in list(sys.modules.keys()):
    if 'chaos' in mod or 'dataset' in mod or 'data.' in mod:
        del sys.modules[mod]

from data.datasetDDP import FSSDataset
FSSDataset.initialize(img_size=400,
    datapath='/kaggle/input/datasets/satvikagada/chaost2')
dl, _ = FSSDataset.build_dataloader('chaost2', 10, 0, 0, 'test', 1)
print(f"Batches: {len(dl)}")


CHAOS: 247 slices found
Batches: 100


In [51]:
torch.cuda.empty_cache()
gc.collect()

utils.fix_randseed(0)
# Replace YOUR_CHAOS_DATASET with actual path found above
FSSDataset.initialize(img_size=400, datapath='/kaggle/input/datasets/satvikagada/chaost2')
dl, _ = FSSDataset.build_dataloader('chaost2', 10, 0, 0, 'test', 1)
avg = AverageMeter(dl.dataset)

with torch.no_grad():
    for idx, batch in enumerate(dl):
        batch = utils.to_cuda(batch)
        pred_mask, _, _ = model.module.predict_mask_nshot(batch, nshot=1)
        assert pred_mask.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred_mask.clone(), batch)
        avg.update(inter, union, batch['class_id'], None)
        if idx % 10 == 0:
            m, _ = avg.compute_iou()
            print(f"Batch {idx+1}/{len(dl)} mIoU: {m:.2f}", flush=True)

miou_chaos_1s, _ = avg.compute_iou()
print(f"\nCHAOS 1-shot: {miou_chaos_1s:.2f}  (paper: 14.3)")

CHAOS: 247 slices found
Batch 1/100 mIoU: 12.01
Batch 11/100 mIoU: 8.57
Batch 21/100 mIoU: 7.83
Batch 31/100 mIoU: 8.14
Batch 41/100 mIoU: 8.03
Batch 51/100 mIoU: 7.84
Batch 61/100 mIoU: 7.77
Batch 71/100 mIoU: 7.81
Batch 81/100 mIoU: 7.98
Batch 91/100 mIoU: 7.88

CHAOS 1-shot: 7.98  (paper: 14.3)


In [52]:
torch.cuda.empty_cache()
gc.collect()

utils.fix_randseed(0)
dl, _ = FSSDataset.build_dataloader('chaost2', 4, 0, 0, 'test', 5)
avg = AverageMeter(dl.dataset)

with torch.no_grad():
    for idx, batch in enumerate(dl):
        batch = utils.to_cuda(batch)
        pred_mask, _, _ = model.module.predict_mask_nshot(batch, nshot=5)
        assert pred_mask.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred_mask.clone(), batch)
        avg.update(inter, union, batch['class_id'], None)
        if idx % 10 == 0:
            m, _ = avg.compute_iou()
            print(f"Batch {idx+1}/{len(dl)} mIoU: {m:.2f}", flush=True)

miou_chaos_5s, _ = avg.compute_iou()
print(f"\nCHAOS 5-shot: {miou_chaos_5s:.2f}  (paper: 17.2)")

CHAOS: 247 slices found
Batch 1/250 mIoU: 2.87
Batch 11/250 mIoU: 7.77
Batch 21/250 mIoU: 7.82
Batch 31/250 mIoU: 7.44
Batch 41/250 mIoU: 7.73
Batch 51/250 mIoU: 7.65
Batch 61/250 mIoU: 7.69
Batch 71/250 mIoU: 7.95
Batch 81/250 mIoU: 7.97
Batch 91/250 mIoU: 7.98
Batch 101/250 mIoU: 7.84
Batch 111/250 mIoU: 7.89
Batch 121/250 mIoU: 7.99
Batch 131/250 mIoU: 8.00
Batch 141/250 mIoU: 8.04
Batch 151/250 mIoU: 8.08
Batch 161/250 mIoU: 8.10
Batch 171/250 mIoU: 8.14
Batch 181/250 mIoU: 8.08
Batch 191/250 mIoU: 8.12
Batch 201/250 mIoU: 8.07
Batch 211/250 mIoU: 8.09
Batch 221/250 mIoU: 8.10
Batch 231/250 mIoU: 8.15
Batch 241/250 mIoU: 8.19

CHAOS 5-shot: 8.16  (paper: 17.2)


In [13]:
cdtg_code = '''r""" DMTNet + CD-TG Integration """
from functools import reduce
from operator import add
import os, sys

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet
import torchvision

from model.base.feature import extract_feat_res
from model.base.correlation import Correlation
from model.learner import HPNLearner

CDTG_DIR = "/kaggle/working/CD_TG"
sys.path.insert(0, CDTG_DIR)
from module import ADaIN, AttentionModule, FFAttention
from encoder import CLIPModel


class DMTNetCDTG(nn.Module):
    def __init__(self, backbone="resnet50", clip_model_name="ViT-B-16",
                 clip_model_path=None, alpha=0.3):
        super(DMTNetCDTG, self).__init__()
        self.alpha    = alpha
        self.cdtg_dir = CDTG_DIR

        self.backbone = resnet.resnet50(
            weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)
        self.feat_ids      = list(range(4, 17))
        self.extract_feats = extract_feat_res
        nbottlenecks       = [3, 4, 6, 3]

        self.reference_layer3 = nn.Linear(2048, 2, bias=True)
        self.reference_layer2 = nn.Linear(1024, 2, bias=True)
        self.reference_layer1 = nn.Linear(512,  2, bias=True)
        self.reference_layer6 = nn.Linear(2048, 2, bias=True)
        self.reference_layer5 = nn.Linear(1024, 2, bias=True)
        self.reference_layer4 = nn.Linear(512,  2, bias=True)
        for layer in [self.reference_layer1, self.reference_layer2,
                      self.reference_layer3, self.reference_layer4,
                      self.reference_layer5, self.reference_layer6]:
            nn.init.kaiming_normal_(layer.weight, a=0, mode="fan_in", nonlinearity="linear")
            nn.init.constant_(layer.bias, 0)

        self.bottleneck_ids = reduce(add, list(map(lambda x: list(range(x)), nbottlenecks)))
        self.lids           = reduce(add, [[i+1]*x for i,x in enumerate(nbottlenecks)])
        self.stack_ids      = torch.tensor(self.lids).bincount().__reversed__().cumsum(dim=0)[:3]
        self.backbone.eval()
        self.hpn_learner        = HPNLearner(list(reversed(nbottlenecks[-3:])))
        self.cross_entropy_loss = nn.CrossEntropyLoss()
        self.dropout            = nn.Dropout(p=0.2)

        self.clip      = CLIPModel(model_name=clip_model_name, model_path=clip_model_path)
        self.proj      = nn.Conv2d(2048, 512, kernel_size=1)
        self.attention = AttentionModule(embed_size=512)
        self.adain     = ADaIN(n_output=512, strength=0.3)
        self.ff_attn   = FFAttention(weight_image=1-alpha, weight_text=alpha)
        self.proj_back = nn.Linear(512, 2048)

    def _get_text_embeddings(self, class_ids, benchmark, device):
        orig_dir = os.getcwd()
        try:
            os.chdir(self.cdtg_dir)
            text_emb    = self.clip.forward(class_ids, benchmark)
            bg_ids      = torch.zeros(class_ids.shape[0], dtype=torch.long).to(device)
            text_emb_bg = self.clip.forward(bg_ids, "background")
        finally:
            os.chdir(orig_dir)
        return text_emb, text_emb_bg

    def forward(self, query_img, support_img, support_mask,
                class_ids=None, benchmark="isic", training=False):

        with torch.no_grad():
            query_feats   = self.extract_feats(query_img,   self.backbone,
                                               self.feat_ids, self.bottleneck_ids, self.lids)
            support_feats = self.extract_feats(support_img, self.backbone,
                                               self.feat_ids, self.bottleneck_ids, self.lids)
            bg_support_feats, prototypes_f_grid, prototypes_b_grid = self.mask_feature_grid(
                support_feats, support_mask.clone(), query_feats)
            support_feats_masked, prototypes_f, prototypes_b = self.mask_feature(
                support_feats, support_mask.clone())

        if class_ids is not None:
            text_emb, _ = self._get_text_embeddings(class_ids, benchmark, query_img.device)
            eps         = 1e-6
            stack_start = len(support_feats_masked) - 3
            for i in range(3):
                idx        = stack_start + i
                s_512      = self.proj(support_feats_masked[idx])
                if training:
                    s_512  = self.adain(s_512)
                s_text     = self.attention(s_512, text_emb)
                mask       = F.interpolate(support_mask.unsqueeze(1).float(),
                                           s_text.size()[2:], mode="bilinear", align_corners=True)
                proto_text = (s_text * mask).sum((2,3)) / (mask.sum((2,3)) + eps)
                proto_2048 = self.proj_back(proto_text)
                prototypes_f[idx] = self.ff_attn(proto_2048, prototypes_f[idx])

        prototypes_f_q, prototypes_b_q, pred_mask = self.query_prototypes(
            query_feats, prototypes_f_grid, prototypes_b_grid)
        query_feats_t, support_feats_t, bg_feats_t = self.Transformation_Feature(
            query_feats, support_feats_masked, bg_support_feats,
            prototypes_f, prototypes_b, prototypes_f_q, prototypes_b_q)

        corr    = Correlation.multilayer_correlation(query_feats_t, support_feats_t, self.stack_ids)
        bg_corr = Correlation.multilayer_correlation(query_feats_t, bg_feats_t,      self.stack_ids)

        logit_mask    = self.hpn_learner(corr)
        logit_mask    = F.interpolate(logit_mask,    support_img.size()[2:], mode="bilinear", align_corners=True)
        bg_logit_mask = self.hpn_learner(bg_corr)
        bg_logit_mask = F.interpolate(bg_logit_mask, support_img.size()[2:], mode="bilinear", align_corners=True)
        return logit_mask, bg_logit_mask, pred_mask

    def get_index(self, i, j, k):
        assert i >= j
        for t in range(0, i - j + 1, k):
            yield t
        if t + j < i:
            yield i - j

    def get_grid(self, shape, grid_num, overlap=False):
        i_h, i_w = shape
        j_h, j_w = i_h//grid_num, i_w//grid_num
        k_h, k_w = (j_h//2, j_w//2) if overlap else (j_h, j_w)
        grids = []
        for h in self.get_index(i_h, j_h, k_h):
            for w in self.get_index(i_w, j_w, k_w):
                grids.append((slice(h, h+j_h), slice(w, w+j_w)))
        return grids

    def mask_feature_grid(self, features, support_mask, query_feats):
        eps = 1e-6
        prototypes_f, prototypes_b, bg_features = [], [], []
        for idx, feature in enumerate(features):
            fg_mask = F.interpolate(support_mask.unsqueeze(1).float(),
                                    feature.size()[2:], mode="bilinear", align_corners=True)
            bg_mask = 1 - fg_mask
            bg_feat = features[idx] * bg_mask
            fg_feat = features[idx] * fg_mask
            bg_features.append(bg_feat)
            grids   = self.get_grid(fg_feat.shape[-2:], grid_num=4)
            grid_pf, grid_pb = [], []
            for grid in grids:
                grid_pf.append(fg_feat[:,:,grid[0],grid[1]].sum((2,3)) /
                               (fg_mask[:,:,grid[0],grid[1]].sum((2,3)) + eps))
                grid_pb.append(bg_feat[:,:,grid[0],grid[1]].sum((2,3)) /
                               (bg_mask[:,:,grid[0],grid[1]].sum((2,3)) + eps))
            prototypes_f.append(grid_pf)
            prototypes_b.append(grid_pb)
        return bg_features, prototypes_f, prototypes_b

    def mask_feature(self, features, support_mask):
        eps = 1e-6
        prototypes_f, prototypes_b, bg_features, mask_features = [], [], [], []
        for idx, feature in enumerate(features):
            mask    = F.interpolate(support_mask.unsqueeze(1).float(),
                                    feature.size()[2:], mode="bilinear", align_corners=True)
            bg_mask = 1 - mask
            bg_feat = features[idx] * bg_mask
            bg_features.append(bg_feat)
            mask_features.append(features[idx] * mask)
            features[idx] = features[idx] * mask
            prototypes_f.append(features[idx].sum((2,3)) / (mask.sum((2,3)) + eps))
            prototypes_b.append(bg_feat.sum((2,3)) / (bg_mask.sum((2,3)) + eps))
        return mask_features, prototypes_f, prototypes_b

    def Transformation_Feature(self, query_feats, support_feats, bg_support_feats,
                                prototypes_f, prototypes_b, prototypes_b_q, prototypes_f_q):
        t_q, t_s, t_bg = [], [], []
        bsz = query_feats[0].shape[0]
        eps = 1e-6
        for idx, feature in enumerate(support_feats):
            C   = torch.cat((prototypes_b[idx].unsqueeze(1), prototypes_f[idx].unsqueeze(1)), dim=1)
            C_q = torch.cat((prototypes_b_q[idx].unsqueeze(1), prototypes_f_q[idx].unsqueeze(1)), dim=1)
            if idx <= 3:
                R, R_q = self.reference_layer1.weight.expand(C.shape), self.reference_layer4.weight.expand(C_q.shape)
            elif idx <= 9:
                R, R_q = self.reference_layer2.weight.expand(C.shape), self.reference_layer5.weight.expand(C_q.shape)
            else:
                R, R_q = self.reference_layer3.weight.expand(C.shape), self.reference_layer6.weight.expand(C_q.shape)
            R, R_q = self.dropout(R), self.dropout(R_q)
            R   = R   / (((R*R).sum(dim=2,keepdim=True)).sqrt()     + eps)
            R_q = R_q / (((R_q*R_q).sum(dim=2,keepdim=True)).sqrt() + eps)
            C   = C   / (((C*C).sum(dim=2,keepdim=True)).sqrt()     + eps)
            C_q = C_q / (((C_q*C_q).sum(dim=2,keepdim=True)).sqrt() + eps)
            P   = torch.matmul(torch.pinverse(C),   R).permute(0,2,1)
            P_q = torch.matmul(torch.pinverse(C_q), R_q).permute(0,2,1)
            P_q = P_q * 0.5 + P * 0.5
            init = query_feats[idx].shape
            query_feats[idx] = query_feats[idx].view(bsz, C_q.size(2), -1)
            t_q.append(torch.matmul(P_q, query_feats[idx]).view(init))
            init = feature.shape
            feature = feature.view(bsz, C.size(2), -1)
            t_s.append(torch.matmul(P, feature).view(init))
            init = bg_support_feats[idx].shape
            bg_support_feats[idx] = bg_support_feats[idx].view(bsz, C.size(2), -1)
            t_bg.append(torch.matmul(P, bg_support_feats[idx]).view(init))
        return t_q, t_s, t_bg

    def query_prototypes(self, query_feats, prototypes_f, prototypes_b):
        result = []
        for idx, qf in enumerate(query_feats):
            bsz,_,h,w = qf.shape
            out = torch.zeros(bsz,2,h,w).float().cuda()
            for i in range(len(prototypes_f[idx])):
                sf = F.cosine_similarity(qf, prototypes_f[idx][i].unsqueeze(-1).unsqueeze(-1), dim=1)
                sb = F.cosine_similarity(qf, prototypes_b[idx][i].unsqueeze(-1).unsqueeze(-1), dim=1)
                out += torch.cat((sb[:,None,...], sf[:,None,...]), dim=1) * 10.0
            result.append(out / float(len(prototypes_f[0])))
        pf_q, pb_q, eps = [], [], 1e-6
        for idx, sim in enumerate(result):
            pm = sim.argmax(dim=1).unsqueeze(1).float()
            bm = 1 - pm
            pf_q.append((query_feats[idx]*pm).sum((2,3)) / (pm.sum((2,3)) + eps))
            pb_q.append((query_feats[idx]*bm).sum((2,3)) / (bm.sum((2,3)) + eps))
        return pf_q, pb_q, result

    def predict_mask_nshot(self, batch, nshot, benchmark="isic"):
        logit_mask_agg = 0
        for s_idx in range(nshot):
            logit_mask, _, _ = self(
                batch["query_img"], batch["support_imgs"][:,s_idx],
                batch["support_masks"][:,s_idx],
                class_ids=batch["class_id"], benchmark=benchmark, training=False)
            logit_mask_agg += logit_mask.argmax(dim=1)
            if nshot == 1:
                return logit_mask_agg, None, None
        bsz      = logit_mask_agg.size(0)
        max_vote = logit_mask_agg.view(bsz,-1).max(dim=1)[0]
        max_vote = torch.stack([max_vote, torch.ones_like(max_vote).long()]).max(dim=0)[0].view(bsz,1,1)
        pred     = logit_mask_agg.float() / max_vote
        pred[pred < 0.5] = 0
        pred[pred >= 0.5] = 1
        return pred, None, None
'''
with open('/kaggle/working/DMTNet/model/dmtnet_cdtg.py', 'w') as f:
    f.write(cdtg_code)
print("dmtnet_cdtg.py written ✓")

dmtnet_cdtg.py written ✓


In [14]:
factory_path = '/usr/local/lib/python3.12/dist-packages/open_clip/factory.py'
with open(factory_path) as f:
    c = f.read()

# Check what's there
for i, line in enumerate(c.split('\n')):
    if 'weights_only' in line or ('torch.load' in line and 'map_location' in line):
        print(f"Line {i}: {line}")

Line 87:     checkpoint = torch.load(checkpoint_path, map_location=map_location)


In [16]:
factory_path = '/usr/local/lib/python3.12/dist-packages/open_clip/factory.py'
with open(factory_path) as f:
    c = f.read()

# Fix the torch.load call in load_state_dict
c = c.replace(
    'checkpoint = torch.load(checkpoint_path, map_location=map_location)',
    'checkpoint = torch.load(checkpoint_path, map_location=map_location, weights_only=False)'
)

with open(factory_path, 'w') as f:
    f.write(c)

# Verify
for i, line in enumerate(c.split('\n')):
    if 'torch.load' in line:
        print(f"Line {i}: {line}")
print("Fixed ✓")


Line 87:     checkpoint = torch.load(checkpoint_path, map_location=map_location, weights_only=False)
Fixed ✓


In [17]:
import importlib
import open_clip
importlib.reload(open_clip.factory)
importlib.reload(open_clip)
print("open_clip reloaded ✓")

open_clip reloaded ✓


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [18]:
import subprocess
# Download proper ViT-B-16 weights in open_clip format
result = subprocess.run(
    'python -c "import open_clip; open_clip.create_model_and_transforms(\'ViT-B-16\', pretrained=\'openai\')"',
    shell=True, capture_output=True, text=True)
print(result.stdout[-500:])
print(result.stderr[-500:])


00, 107MiB/s]
 87%|██████████████████████████████████▋     | 305M/351M [00:02<00:00, 112MiB/s]
 90%|████████████████████████████████████    | 316M/351M [00:03<00:00, 111MiB/s]
 93%|█████████████████████████████████████▎  | 327M/351M [00:03<00:00, 102MiB/s]
 96%|██████████████████████████████████████▍ | 337M/351M [00:03<00:00, 100MiB/s]
 99%|██████████████████████████████████████▌| 347M/351M [00:03<00:00, 91.6MiB/s]
100%|████████████████████████████████████████| 351M/351M [00:03<00:00, 102MiB/s]



In [19]:
# Patch encoder.py to use openai pretrained instead of local file
encoder_path = '/kaggle/working/CD_TG/encoder.py'
with open(encoder_path) as f:
    c = f.read()

# Replace load_open_clip method
old = '''    def load_open_clip(self):
        """Loading the OpenCLIP Local Model"""
        model, preprocess, _ = open_clip.create_model_and_transforms(
            model_name=self.model_name,
            pretrained=self.model_path,
        )'''

new = '''    def load_open_clip(self):
        """Loading the OpenCLIP Local Model"""
        import os
        if self.model_path and os.path.exists(self.model_path):
            try:
                model, preprocess, _ = open_clip.create_model_and_transforms(
                    model_name=self.model_name,
                    pretrained=self.model_path,
                )
            except Exception:
                # Fallback to downloading from openai
                model, preprocess, _ = open_clip.create_model_and_transforms(
                    model_name=self.model_name,
                    pretrained='openai',
                )
        else:
            model, preprocess, _ = open_clip.create_model_and_transforms(
                model_name=self.model_name,
                pretrained='openai',
            )'''

c = c.replace(old, new)
with open(encoder_path, 'w') as f:
    f.write(c)
print("encoder.py fixed ✓")

encoder.py fixed ✓


In [20]:
factory_path = '/usr/local/lib/python3.12/dist-packages/open_clip/factory.py'
with open(factory_path) as f:
    c = f.read()

# Fix load_state_dict to handle TorchScript files
old = '''def load_state_dict(checkpoint_path: str, map_location='cpu'):
    checkpoint = torch.load(checkpoint_path, map_location=map_location)
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint
    if next(iter(state_dict.items()))[0].startswith('module'):
        state_dict = {k[7:]: v for k, v in state_dict.items()}
    return state_dict'''

new = '''def load_state_dict(checkpoint_path: str, map_location='cpu'):
    checkpoint = torch.load(checkpoint_path, map_location=map_location, weights_only=False)
    if torch.jit.isinstance(checkpoint, torch.jit.ScriptModule) or isinstance(checkpoint, torch.jit.ScriptModule):
        return checkpoint.state_dict()
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    elif isinstance(checkpoint, dict):
        state_dict = checkpoint
    else:
        try:
            state_dict = checkpoint.state_dict()
        except Exception:
            state_dict = checkpoint
    if isinstance(state_dict, dict) and len(state_dict) > 0:
        first_key = next(iter(state_dict.keys()))
        if first_key.startswith('module'):
            state_dict = {k[7:]: v for k, v in state_dict.items()}
    return state_dict'''

c = c.replace(old, new)
with open(factory_path, 'w') as f:
    f.write(c)
print("Fixed ✓")

Fixed ✓


In [21]:
factory_path = '/usr/local/lib/python3.12/dist-packages/open_clip/factory.py'
with open(factory_path) as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if 'checkpoint = torch.load(checkpoint_path, map_location=map_location)' in line:
        line = line.replace(
            'checkpoint = torch.load(checkpoint_path, map_location=map_location)',
            'checkpoint = torch.load(checkpoint_path, map_location=map_location, weights_only=False)'
        )
    if "if next(iter(state_dict.items()))[0].startswith('module'):" in line:
        new_lines.append('    if isinstance(state_dict, dict) and len(state_dict) > 0 and next(iter(state_dict.keys())).startswith("module"):\n')
        continue
    new_lines.append(line)

with open(factory_path, 'w') as f:
    f.writelines(new_lines)
print("Fixed ✓")

Fixed ✓


In [23]:
import subprocess
r = subprocess.run(
    'python -c "import open_clip; m,_,_ = open_clip.create_model_and_transforms(\'ViT-B-16\', pretrained=\'openai\'); import torch; torch.save(m.state_dict(), \'/kaggle/working/clip_weights/ViT-B-16-openai.pt\')"',
    shell=True, capture_output=True, text=True)
print(r.stdout[-500:])
print(r.stderr[-500:])
import os
if os.path.exists('/kaggle/working/clip_weights/ViT-B-16-openai.pt'):
    print(f"Saved: {os.path.getsize('/kaggle/working/clip_weights/ViT-B-16-openai.pt')/1e6:.1f} MB")


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)

Saved: 598.6 MB


In [31]:
with open('/kaggle/working/CD_TG/encoder.py', 'r') as f:
    print(f.read()[:500])   # print first 1000 chars

In [32]:
encoder_code = '''
import torch
import open_clip

class CLIPModel:
    def __init__(self, model_name="ViT-B-16", model_path=None):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model, _, self.preprocess = open_clip.create_model_and_transforms(
            model_name=model_name,
            pretrained="openai"
        )
        self.tokenizer = open_clip.get_tokenizer(model_name)
        self.model = self.model.to(self.device)
        self.model.eval()

    def forward(self, category_ids, model):
        # Simple text prompts
        texts = [f"a photo of organ {int(c.item())}" for c in category_ids]
        text_tokens = self.tokenizer(texts).to(self.device)

        with torch.no_grad():
            text_features = self.model.encode_text(text_tokens)

        return text_features, None
'''

with open('/kaggle/working/CD_TG/encoder.py', 'w') as f:
    f.write(encoder_code)

print("encoder.py RESET ✅")

encoder.py RESET ✅


In [33]:
import importlib
import sys

if "encoder" in sys.modules:
    del sys.modules["encoder"]

import model.dmtnet_cdtg
importlib.reload(model.dmtnet_cdtg)

print("Reloaded clean ✅")

Reloaded clean ✅


In [35]:
# Reload with prefix fix
src = torch.load('/kaggle/working/DMTNet/dmtnet_resnet50.pt',
                 map_location=device, weights_only=False)

# Check key format
sample_keys = list(src.keys())[:5]
print("Source keys sample:", sample_keys)

model_dict = cdtg.state_dict()
sample_model_keys = list(model_dict.keys())[:5]
print("Model keys sample:", sample_model_keys)

Source keys sample: ['module.backbone.conv1.weight', 'module.backbone.bn1.weight', 'module.backbone.bn1.bias', 'module.backbone.bn1.running_mean', 'module.backbone.bn1.running_var']
Model keys sample: ['module.backbone.conv1.weight', 'module.backbone.bn1.weight', 'module.backbone.bn1.bias', 'module.backbone.bn1.running_mean', 'module.backbone.bn1.running_var']


In [37]:
import gc
torch.cuda.empty_cache()
gc.collect()

# Rebuild cdtg fresh - NOT wrapped in DataParallel yet
for mod in list(sys.modules.keys()):
    if 'dmtnet_cdtg' in mod:
        del sys.modules[mod]

from model.dmtnet_cdtg import DMTNetCDTG

cdtg = DMTNetCDTG(backbone='resnet50', clip_model_name='ViT-B-16',
                   clip_model_path='/kaggle/working/clip_weights/ViT-B-16-openai.pt',
                   alpha=0.3)

# Load weights
src = torch.load('/kaggle/working/DMTNet/dmtnet_resnet50.pt',
                 map_location=device, weights_only=False)

# Remove module. prefix from src
src_clean = {k.replace('module.', '', 1): v for k, v in src.items()}

model_dict = cdtg.state_dict()
print("Model keys sample:", list(model_dict.keys())[:3])
print("Src keys sample:",   list(src_clean.keys())[:3])

matched = {k:v for k,v in src_clean.items()
           if k in model_dict and model_dict[k].shape == v.shape}
print(f"Matched: {len(matched)}/{len(model_dict)}")

model_dict.update(matched)
cdtg.load_state_dict(model_dict)

# NOW wrap in DataParallel
cdtg = nn.DataParallel(cdtg)
cdtg.to(device).eval()
print("CD-TG loaded ✓")


Model keys sample: ['backbone.conv1.weight', 'backbone.bn1.weight', 'backbone.bn1.bias']
Src keys sample: ['backbone.conv1.weight', 'backbone.bn1.weight', 'backbone.bn1.bias']
Matched: 430/444
CD-TG loaded ✓


In [38]:
import torch

# Load as TorchScript
ts_model = torch.jit.load('/kaggle/working/clip_weights/ViT-B-16.pt',
                           map_location='cpu')
print("TorchScript loaded ✓")
print(type(ts_model))

# Save as regular state dict
torch.save(ts_model.state_dict(),
           '/kaggle/working/clip_weights/ViT-B-16-converted.pt')
import os
print(f"Saved: {os.path.getsize('/kaggle/working/clip_weights/ViT-B-16-converted.pt')/1e6:.1f} MB")


TorchScript loaded ✓
<class 'torch.jit._script.RecursiveScriptModule'>
Saved: 350.5 MB


In [39]:
torch.cuda.empty_cache()
import gc; gc.collect()

for mod in list(sys.modules.keys()):
    if 'dmtnet_cdtg' in mod or 'encoder' in mod:
        del sys.modules[mod]

from model.dmtnet_cdtg import DMTNetCDTG

cdtg = DMTNetCDTG(backbone='resnet50', clip_model_name='ViT-B-16',
                   clip_model_path='/kaggle/working/clip_weights/ViT-B-16-converted.pt',
                   alpha=0.3)

src = torch.load('/kaggle/working/DMTNet/dmtnet_resnet50.pt',
                 map_location=device, weights_only=False)
src_clean = {k.replace('module.', '', 1): v for k, v in src.items()}
model_dict = cdtg.state_dict()
matched = {k:v for k,v in src_clean.items()
           if k in model_dict and model_dict[k].shape == v.shape}
model_dict.update(matched)
cdtg.load_state_dict(model_dict)
cdtg = nn.DataParallel(cdtg)
cdtg.to(device).eval()
print(f"CD-TG loaded ✓  matched {len(matched)}/{len(model_dict)}")

CD-TG loaded ✓  matched 430/444


In [73]:
torch.cuda.empty_cache(); gc.collect()
utils.fix_randseed(0)
FSSDataset.initialize(img_size=400, datapath='/kaggle/working')
dl, _ = FSSDataset.build_dataloader('isic', 10, 0, 0, 'test', 1)
avg = AverageMeter(dl.dataset)
with torch.no_grad():
    for idx, batch in enumerate(dl):
        batch = utils.to_cuda(batch)
        pred_mask, _, _ = cdtg.module.predict_mask_nshot(batch, nshot=1, benchmark='isic')
        assert pred_mask.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred_mask.clone(), batch)
        avg.update(inter, union, batch['class_id'], None)
        if idx % 10 == 0:
            m, _ = avg.compute_iou()
            print(f"Batch {idx+1}/{len(dl)} mIoU: {m:.2f}", flush=True)
miou_converted, _ = avg.compute_iou()
print(f"\nISIC 1-shot with converted weights: {miou_converted:.2f}")
print(f"ISIC 1-shot with openai weights:    {miou_cdtg_isic_1s:.2f}")
print(f"Paper CD-TG:                         47.4")

Total (test) isic images are : 600
Batch 1/60 mIoU: 54.39
Batch 11/60 mIoU: 40.50
Batch 21/60 mIoU: 40.57
Batch 31/60 mIoU: 39.78
Batch 41/60 mIoU: 40.80
Batch 51/60 mIoU: 41.59

ISIC 1-shot with converted weights: 42.35
ISIC 1-shot with openai weights:    42.39
Paper CD-TG:                         47.4


In [74]:
torch.cuda.empty_cache(); gc.collect()
utils.fix_randseed(0)
dl, _ = FSSDataset.build_dataloader('isic', 4, 0, 0, 'test', 5)
avg = AverageMeter(dl.dataset)
with torch.no_grad():
    for idx, batch in enumerate(dl):
        batch = utils.to_cuda(batch)
        pred_mask, _, _ = cdtg.module.predict_mask_nshot(batch, nshot=5, benchmark='isic')
        assert pred_mask.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred_mask.clone(), batch)
        avg.update(inter, union, batch['class_id'], None)
        if idx % 10 == 0:
            m, _ = avg.compute_iou()
            print(f"Batch {idx+1}/{len(dl)} mIoU: {m:.2f}", flush=True)
miou_cdtg_isic_5s, _ = avg.compute_iou()
print(f"\nCD-TG ISIC 5-shot: {miou_cdtg_isic_5s:.2f}  (paper: 53.7)")

Total (test) isic images are : 600
Batch 1/150 mIoU: 67.50
Batch 11/150 mIoU: 49.70
Batch 21/150 mIoU: 52.15
Batch 31/150 mIoU: 52.08
Batch 41/150 mIoU: 51.32
Batch 51/150 mIoU: 50.93
Batch 61/150 mIoU: 50.51
Batch 71/150 mIoU: 51.17
Batch 81/150 mIoU: 51.09
Batch 91/150 mIoU: 50.58
Batch 101/150 mIoU: 50.69
Batch 111/150 mIoU: 50.89
Batch 121/150 mIoU: 51.39
Batch 131/150 mIoU: 50.62
Batch 141/150 mIoU: 50.37

CD-TG ISIC 5-shot: 50.81  (paper: 53.7)


In [75]:
torch.cuda.empty_cache(); gc.collect()
utils.fix_randseed(0)
FSSDataset.initialize(img_size=400,
    datapath='/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels')
dl, _ = FSSDataset.build_dataloader('lung', 10, 0, 0, 'test', 1)
avg = AverageMeter(dl.dataset)
with torch.no_grad():
    for idx, batch in enumerate(dl):
        batch = utils.to_cuda(batch)
        pred_mask, _, _ = cdtg.module.predict_mask_nshot(batch, nshot=1, benchmark='lung')
        assert pred_mask.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred_mask.clone(), batch)
        avg.update(inter, union, batch['class_id'], None)
        if idx % 5 == 0:
            m, _ = avg.compute_iou()
            print(f"Batch {idx+1}/{len(dl)} mIoU: {m:.2f}", flush=True)
miou_cdtg_lung_1s, _ = avg.compute_iou()
print(f"\nCD-TG Lung 1-shot: {miou_cdtg_lung_1s:.2f}  (paper: 77.8)")

Lung: 566 matched pairs
Batch 1/60 mIoU: 61.02
Batch 6/60 mIoU: 60.99
Batch 11/60 mIoU: 62.11
Batch 16/60 mIoU: 60.77
Batch 21/60 mIoU: 61.80
Batch 26/60 mIoU: 61.37
Batch 31/60 mIoU: 61.07
Batch 36/60 mIoU: 61.27
Batch 41/60 mIoU: 61.02
Batch 46/60 mIoU: 60.79
Batch 51/60 mIoU: 60.76
Batch 56/60 mIoU: 60.81

CD-TG Lung 1-shot: 60.83  (paper: 77.8)


In [76]:
torch.cuda.empty_cache(); gc.collect()
utils.fix_randseed(0)
dl, _ = FSSDataset.build_dataloader('lung', 4, 0, 0, 'test', 5)
avg = AverageMeter(dl.dataset)
with torch.no_grad():
    for idx, batch in enumerate(dl):
        batch = utils.to_cuda(batch)
        pred_mask, _, _ = cdtg.module.predict_mask_nshot(batch, nshot=5, benchmark='lung')
        assert pred_mask.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred_mask.clone(), batch)
        avg.update(inter, union, batch['class_id'], None)
        if idx % 10 == 0:
            m, _ = avg.compute_iou()
            print(f"Batch {idx+1}/{len(dl)} mIoU: {m:.2f}", flush=True)
miou_cdtg_lung_5s, _ = avg.compute_iou()
print(f"\nCD-TG Lung 5-shot: {miou_cdtg_lung_5s:.2f}  (paper: 79.4)")

Lung: 566 matched pairs
Batch 1/150 mIoU: 62.94
Batch 11/150 mIoU: 60.72
Batch 21/150 mIoU: 62.02
Batch 31/150 mIoU: 61.94
Batch 41/150 mIoU: 61.95
Batch 51/150 mIoU: 62.24
Batch 61/150 mIoU: 61.79
Batch 71/150 mIoU: 61.59
Batch 81/150 mIoU: 61.96
Batch 91/150 mIoU: 62.36
Batch 101/150 mIoU: 62.39
Batch 111/150 mIoU: 62.18
Batch 121/150 mIoU: 62.00
Batch 131/150 mIoU: 61.82
Batch 141/150 mIoU: 61.62

CD-TG Lung 5-shot: 61.65  (paper: 79.4)


In [77]:
torch.cuda.empty_cache(); gc.collect()
utils.fix_randseed(0)
FSSDataset.initialize(img_size=400,
    datapath='/kaggle/working/wbc_organized/segmentation_WBC-master')
dl, _ = FSSDataset.build_dataloader('wbc', 10, 0, 0, 'test', 1)
avg = AverageMeter(dl.dataset)
with torch.no_grad():
    for idx, batch in enumerate(dl):
        batch = utils.to_cuda(batch)
        pred_mask, _, _ = cdtg.module.predict_mask_nshot(batch, nshot=1, benchmark='wbc')
        assert pred_mask.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred_mask.clone(), batch)
        avg.update(inter, union, batch['class_id'], None)
        if idx % 10 == 0:
            m, _ = avg.compute_iou()
            print(f"Batch {idx+1}/{len(dl)} mIoU: {m:.2f}", flush=True)
miou_cdtg_wbc_1s, _ = avg.compute_iou()
print(f"\nCD-TG WBC 1-shot: {miou_cdtg_wbc_1s:.2f}  (paper: 50.8)")

WBC: 400 pairs found
WBC img_path: /kaggle/working/wbc_organized/segmentation_WBC-master/images
Batch 1/100 mIoU: 85.75
Batch 11/100 mIoU: 80.97
Batch 21/100 mIoU: 78.94
Batch 31/100 mIoU: 80.20
Batch 41/100 mIoU: 79.53
Batch 51/100 mIoU: 78.61
Batch 61/100 mIoU: 78.60
Batch 71/100 mIoU: 78.92
Batch 81/100 mIoU: 79.10
Batch 91/100 mIoU: 79.25

CD-TG WBC 1-shot: 79.78  (paper: 50.8)


In [78]:
torch.cuda.empty_cache(); gc.collect()
utils.fix_randseed(0)
dl, _ = FSSDataset.build_dataloader('wbc', 4, 0, 0, 'test', 5)
avg = AverageMeter(dl.dataset)
with torch.no_grad():
    for idx, batch in enumerate(dl):
        batch = utils.to_cuda(batch)
        pred_mask, _, _ = cdtg.module.predict_mask_nshot(batch, nshot=5, benchmark='wbc')
        assert pred_mask.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred_mask.clone(), batch)
        avg.update(inter, union, batch['class_id'], None)
        if idx % 10 == 0:
            m, _ = avg.compute_iou()
            print(f"Batch {idx+1}/{len(dl)} mIoU: {m:.2f}", flush=True)
miou_cdtg_wbc_5s, _ = avg.compute_iou()
print(f"\nCD-TG WBC 5-shot: {miou_cdtg_wbc_5s:.2f}  (paper: 52.5)")

WBC: 400 pairs found
WBC img_path: /kaggle/working/wbc_organized/segmentation_WBC-master/images
Batch 1/250 mIoU: 75.46
Batch 11/250 mIoU: 85.40
Batch 21/250 mIoU: 83.22
Batch 31/250 mIoU: 83.79
Batch 41/250 mIoU: 84.25
Batch 51/250 mIoU: 84.74
Batch 61/250 mIoU: 84.42
Batch 71/250 mIoU: 84.22
Batch 81/250 mIoU: 84.41
Batch 91/250 mIoU: 84.20
Batch 101/250 mIoU: 84.59
Batch 111/250 mIoU: 84.77
Batch 121/250 mIoU: 85.29
Batch 131/250 mIoU: 85.12
Batch 141/250 mIoU: 84.91
Batch 151/250 mIoU: 85.11
Batch 161/250 mIoU: 85.12
Batch 171/250 mIoU: 85.22
Batch 181/250 mIoU: 85.40
Batch 191/250 mIoU: 85.24
Batch 201/250 mIoU: 85.27
Batch 211/250 mIoU: 85.10
Batch 221/250 mIoU: 85.19
Batch 231/250 mIoU: 85.12
Batch 241/250 mIoU: 85.29

CD-TG WBC 5-shot: 85.29  (paper: 52.5)
